# 04 — Hazard heads on frozen ClimaX encoders (Option A)

**Question:** does a physics-informed encoder (λ = 0.001) give better extreme-event detection than a data-only encoder (λ = 0)
when both feed identical hazard heads? And do either beat simple baselines?

* Hazards: heatwave (Mar–Jun), extreme 10 m wind, extreme precipitation (IMD); definitions in `docs/mtech/HAZARD_DEFINITIONS.md`.
* Task: forecast issued at 00 UTC on day D from the fields at that hour → hazard on day D+3 (72 h lead, same as the encoder).
* Encoders are **frozen**; only small heads are trained (3 seeds each). Baselines: climatology, persistence, logistic regression.
* Grid: the original 6×5 dataset. The 6×6 internship models get the in-memory edge pad; the duplicated column is never trained or scored.
* Thresholds are chosen on val (max F1) and applied to test. Metrics: PR-AUC (primary), ROC-AUC, F1, precision, recall, confusion counts.

**Attach:** `climax-indian-monsoon-regional`, `checkpoints-for-regional-forecasting` (λ = 0.001), your λ = 0 checkpoint dataset,
and the IMD rainfall dataset (`rain/1979.grd` … `2018.grd`). Runs on CPU in roughly 20–40 min; a GPU makes feature extraction faster.

In [ ]:
INPUT_ROOT = "/kaggle/input"
DATA_DIR_OVERRIDE = None        # original 6x5 regional dataset
IMD_DIR_OVERRIDE = None         # folder containing rain/<year>.grd
PHYSICS_CKPT_OVERRIDE = None    # e.g. ".../checkpoints-for-regional-forecasting/epoch_016.ckpt"
NOPHYSICS_CKPT_OVERRIDE = None  # your lambda = 0 best checkpoint
SEEDS = [0, 1, 2]
EPOCHS = 40
SMOKE_TEST = False              # True: 300 samples per split, 2 epochs, 1 seed (pipeline check, ~3 min)
REPO_URL = "https://github.com/Aditya1Agrawal/ClimaX-Monsoon-PINN.git"
REPO_BRANCH = "mtech-extreme-events"
OUT_DIR = "/kaggle/working/hazard_optionA" + ("_smoke" if SMOKE_TEST else "")

In [ ]:
import os, sys, json, subprocess, pathlib, importlib.util
REPO_DIR = "/kaggle/working/ClimaX-Monsoon-PINN"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
print("repo commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "timm==0.6.12"], check=True)
timm_init = pathlib.Path(importlib.util.find_spec("timm").origin).parent / "models" / "__init__.py"
txt = timm_init.read_text()
if sys.version_info >= (3, 11) and "\nfrom .maxxvit import *" in txt:
    timm_init.write_text(txt.replace("\nfrom .maxxvit import *", "\n# from .maxxvit import *  # disabled for py>=3.11"))
sys.path.insert(0, f"{REPO_DIR}/src")
import torch, numpy as np, pandas as pd
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## Discover inputs and identify the checkpoints from their own metadata

In [ ]:
from climax_hazards import inspect_utils as iu
cands = [p for p in iu.find_dataset_dirs(INPUT_ROOT)
         if np.load(p / "lon.npy").size == 5 and not iu.detect_lon_padding(p)["looks_padded"]]
DATA_DIR = iu.resolve_single(cands, DATA_DIR_OVERRIDE, "original 6x5 dataset")

def find_imd(root):
    # folder holding 1979.grd ... 2018.grd (flat upload or imdlib's rain/ subfolder)
    for dirpath, _, files in os.walk(root):
        if "1979.grd" in files:
            return dirpath
IMD_DIR = IMD_DIR_OVERRIDE or find_imd(INPUT_ROOT)
print("DATA_DIR:", DATA_DIR, "\nIMD_DIR:", IMD_DIR or "NOT FOUND - precipitation will be skipped")

rows = []
for p in iu.find_checkpoints(INPUT_ROOT):
    ck = iu.load_checkpoint(p)
    mc = next((v for k, v in (ck.get("callbacks") or {}).items() if k.startswith("ModelCheckpoint")), {})
    best_name = os.path.basename(mc.get("best_model_path", "") or "")
    rows.append({"path": str(p), "run_dir": str(p.parent), "file": p.name,
                 "lambda": ck.get("hyper_parameters", {}).get("physics_lambda"),
                 "epoch": ck.get("epoch"), "best_at_save": best_name,
                 "best_val_w_rmse": float(mc["best_model_score"]) if mc.get("best_model_score") is not None else None})
    del ck
ckpts = pd.DataFrame(rows)
# Each checkpoint stores the best file *at the time it was saved*; the run's final choice is the
# one recorded in its latest checkpoint (highest epoch, normally last.ckpt).
final = ckpts.sort_values("epoch").groupby("run_dir").tail(1).set_index("run_dir")
ckpts["run_final_best"] = [r.file == final.loc[r.run_dir, "best_at_save"] for r in ckpts.itertuples()]
print(ckpts.drop(columns=["path", "run_dir"]).to_string(index=False))

def pick(lam, override):
    if override:
        return os.path.abspath(override)
    sel = ckpts[(ckpts["lambda"] == lam) & ckpts["run_final_best"]]
    if len(sel) != 1:
        raise RuntimeError(f"expected exactly one final-best checkpoint with lambda={lam}, found {len(sel)}; set the override")
    return os.path.abspath(sel["path"].iloc[0])
PHYSICS_CKPT = pick(0.001, PHYSICS_CKPT_OVERRIDE)
NOPHYSICS_CKPT = pick(0.0, NOPHYSICS_CKPT_OVERRIDE)
print("\nphysics  :", PHYSICS_CKPT, "\nnophysics:", NOPHYSICS_CKPT)

## Run the experiment

In [ ]:
import re, time
OUT_DIR = os.path.abspath(OUT_DIR)
cmd = [sys.executable, "scripts/mtech/run_hazard_experiment.py", "--data_dir", os.path.abspath(DATA_DIR),
       "--config", "configs/regional_india.yaml", "--out_dir", OUT_DIR,
       "--checkpoint", f"physics={PHYSICS_CKPT}", "--checkpoint", f"nophysics={NOPHYSICS_CKPT}",
       "--seeds", *map(str, SEEDS), "--epochs", str(EPOCHS)]
if IMD_DIR:
    cmd += ["--imd_dir", os.path.abspath(IMD_DIR)]
if SMOKE_TEST:
    cmd += ["--max_samples", "300", "--epochs", "2", "--seeds", "0"]
os.makedirs(OUT_DIR, exist_ok=True)
t0 = time.time()
with open(f"{OUT_DIR}/run.log", "w") as log:
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src"),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        print(line, end="", flush=True)
    rc = proc.wait()
print(f"exit {rc} after {(time.time() - t0) / 60:.1f} min")
assert rc == 0

## Results (test split, thresholds from val)

In [ ]:
summ = pd.read_csv(f"{OUT_DIR}/hazard_summary_test.csv")
order = ["climatology", "persistence", "logistic", "nophysics", "physics"]
summ["source"] = pd.Categorical(summ["source"], order, ordered=True)
summ = summ.sort_values(["hazard", "source"])
print(summ.round(4).to_string(index=False))

import matplotlib.pyplot as plt
hz = list(summ["hazard"].unique())
fig, axes = plt.subplots(1, len(hz), figsize=(5 * len(hz), 4))
for ax, h in zip(np.atleast_1d(axes), hz):
    s = summ[summ.hazard == h]
    ax.bar(s["source"].astype(str), s["pr_auc"], yerr=s["pr_auc_sd"].fillna(0), capsize=4)
    ax.axhline(s["prevalence"].iloc[0], ls="--", lw=1, color="k", label="prevalence (no-skill PR-AUC)")
    ax.set_title(f"{h}: test PR-AUC"); ax.tick_params(axis="x", rotation=30); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/test_pr_auc.png", dpi=120); plt.show()
print(sorted(os.listdir(OUT_DIR)))